# LCBO Toy Example

This notebook creates a small synthetic constrained optimization problem and passes every optimizer argument directly as Python objects. No YAML configuration is used.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import torch
from gpytorch.constraints import GreaterThan
from gpytorch.kernels import RBFKernel
from gpytorch.priors import GammaPrior

from lcbo_opt import LCBO, generate_initial_data, make_benchmark_problem

In [ ]:
# device: hardware used for tensors and GP models.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# dtype: floating-point precision used throughout the optimizer.
dtype = torch.double

# noise_variance: observation-noise variance used by the synthetic benchmark.
noise_variance = 1e-2

# problem: benchmark object containing objective, constraints, dimension, bounds, and metadata.
problem = make_benchmark_problem(
    "within_model",
    dim=3,
    n_constraints=1,
    seed=0,
    offset=0.5,
    noise_variance=noise_variance,
    device=device,
    dtype=dtype,
)

In [ ]:
# step_size_fn: function that maps the iteration index k to the descent step size.
step_size_fn = lambda k: 0.25

# rho_fn: function that maps the iteration index k to the constraint-penalty weight.
rho_fn = lambda k: 10.0 * (k + 1) ** 0.25

# kernel_factory: function that receives input_dim and returns the GP kernel object.
kernel_factory = lambda input_dim: RBFKernel(ard_num_dims=input_dim)

# prior_config: GP hyperparameter configuration passed as live Python objects.
# GreaterThan and GammaPrior are class instances, which is why direct Python passing is used.
common_gp_config = {
    "lengthscale": {
        "constraint": GreaterThan(1e-3),
        "prior": GammaPrior(concentration=3.0, rate=3.0),
        "base_init": 1.0,
    },
    "outputscale": {
        "constraint": GreaterThan(1e-3),
        "prior": GammaPrior(concentration=2.0, rate=1.0),
    },
    "noise": {"fixed": noise_variance},
}
prior_config = {
    "obj": common_gp_config,
    "cons_0": common_gp_config,
}

In [ ]:
optimizer = LCBO(
    # obj_func: callable objective function to minimize.
    obj_func=problem.objective,
    # constraint_funcs: list of callable constraints; feasible points satisfy g_i(x) <= 0.
    constraint_funcs=problem.constraints,
    # input_dim: number of decision variables.
    input_dim=problem.dim,
    # bounds: tensor with lower bounds in row 0 and upper bounds in row 1.
    bounds=problem.bounds,
    # max_objective_calls: total budget for objective evaluations.
    max_objective_calls=30,
    # n_repeats_gradient: number of repeated observations at the current iterate.
    n_repeats_gradient=1,
    # max_active_points: number of active-sampling points added before each update.
    max_active_points=1,
    # N_max: maximum number of recent observations used to fit each GP.
    N_max=12,
    # step_size_fn: Python function controlling the optimizer step size.
    step_size_fn=step_size_fn,
    # rho_fn: Python function controlling the constraint penalty schedule.
    rho_fn=rho_fn,
    # temp_lse: smoothing temperature for the log-sum-exp acquisition aggregation.
    temp_lse=1.0,
    # prior_config: Python dict containing GP priors, constraints, initial values, and fixed values.
    prior_config=prior_config,
    # set_hyper: True keeps the configured GP hyperparameters fixed instead of periodically optimizing them.
    set_hyper=True,
    # grad_normalize: True normalizes the descent direction before applying step_size_fn.
    grad_normalize=True,
    # delta_norm: trust-region radius in normalized coordinates for active sampling.
    delta_norm=0.1,
    # kernel_factory: Python function that builds the kernel used by every GP model.
    kernel_factory=kernel_factory,
    # pull_gp_param_from_problem: True would replace prior_config with fixed GP parameters from problem.metadata.
    pull_gp_param_from_problem=False,
    # noise_variance: noise level used if pull_gp_param_from_problem=True.
    noise_variance=noise_variance,
    # device: hardware target for tensors and models.
    device=device,
    # dtype: floating-point precision for tensors and models.
    dtype=dtype,
)

In [ ]:
# n0: number of random initial observations before model-based optimization starts.
theta = generate_initial_data(
    optimizer=optimizer,
    objective_func=problem.objective,
    constraint_funcs=problem.constraints,
    bounds=problem.bounds,
    n0=4,
    seed=0,
)

for iteration in range(3):
    theta, stop = optimizer.run_step(theta, iteration)
    if stop:
        break

objective_value = problem.objective(theta, obs_noise=False).item()
constraint_values = [constraint(theta, obs_noise=False).item() for constraint in problem.constraints]

print("Final theta:", theta.detach().cpu().numpy())
print("Objective:", objective_value)
print("Constraints:", constraint_values)
print("Objective calls:", optimizer.call_counter)

If you want to use the fixed GP hyperparameters stored in the synthetic problem metadata, create the optimizer with `problem=problem`, `pull_gp_param_from_problem=True`, and omit `prior_config`.